<a href="https://colab.research.google.com/github/radovanpopovic02/metode_analize_geoprostornih_podataka_domaci/blob/main/kodIspit_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prosečne godišnje padavine u Srbiji (ERA5-Land)

Ova verzija notebook-a računa **prosečne godišnje padavine** za period **2000–2024** i omogućava export slike na **Google Drive**.

Logika:
- uzimaju se mesečne padavine iz ERA5-Land
- za svaku godinu sabiraju se svi meseci
- zatim se računa prosek tih godišnjih suma za ceo period
- rezultat se eksportuje kao GeoTIFF na Drive


In [4]:
import ee
import geemap

# Pokrenuti autentifikaciju samo prvi put ili kada istekne sesija.
ee.Authenticate()
ee.Initialize(project="ee-popovicradovan02")

print("geemap verzija:", geemap.__version__)


geemap verzija: 0.35.3


In [12]:
# Mesečne padavine iz ERA5-Land
period = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterDate("2000-01-01", "2024-12-31")
    .select("total_precipitation_sum")
)

# Granica Srbije
granica1 = (
    ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    .filter(ee.Filter.eq("country_na", "Serbia"))
    .geometry()
)

# Kosovo
granica2 = (
    ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    .filter(ee.Filter.eq("country_na", "Kosovo"))
    .geometry()
)

# Spajanje geometrija
graniceKonacno = granica1.union(granica2)

# Lista godina
years = ee.List.sequence(2000, 2024)

# Godišnja suma padavina za svaku godinu (u mm)
def annual_sum(year):
    year = ee.Number(year)
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    annual_image = (
        period
        .filterDate(start, end)
        .sum()
        .multiply(1000)
        .clip(graniceKonacno)
        .set("year", year)
    )
    return annual_image

annual_collection = ee.ImageCollection(years.map(annual_sum))

# Prosek godišnjih suma padavina za ceo period 2000-2024 (mm/god)
mean_annual_precip = annual_collection.mean()

viz_params = {
    "min": 400,
    "max": 1200,
    "palette": ["white", "lightblue", "blue", "darkblue"]
}

mean_annual_precip


In [15]:


# Pretvaranje geometrije u FeatureCollection
granice_fc = ee.FeatureCollection([ee.Feature(graniceKonacno)])

# Prikaz na mapi
m = geemap.Map(center=[44.0, 20.8], zoom=7)
m.addLayer(mean_annual_precip, viz_params, "Prosečne godišnje padavine 2000-2024 (mm/god)")
m.addLayer(granice_fc.style(**{"color": "red", "fillColor": "#00000000", "width": 2}), {}, "Granica Srbije")
m

Map(center=[44.0, 20.8], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(…

In [19]:
# Export slike na Google Drive kao GeoTIFF
task = ee.batch.Export.image.toDrive(
    image=mean_annual_precip,
    description="Godisnje_Padavine_Srbija_2000_2024",
    folder="GEE_Export",
    fileNamePrefix="godisnje_padavine_srbija_2000_2024",
    region=granice_fc.geometry(),
    scale=10000,
    crs="EPSG:4326",
    fileFormat="GeoTIFF",
    maxPixels=1e13,
)

task.start()
print("Export je pokrenut. Proveri karticu Tasks u Earth Engine/Colab okruženju i folder GEE_Export na Google Drive-u.")


Export je pokrenut. Proveri karticu Tasks u Earth Engine/Colab okruženju i folder GEE_Export na Google Drive-u.


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Opcionalno: prosečne godišnje padavine po opštinama
opstine = (
    ee.FeatureCollection("FAO/GAUL/2015/level2")
    .filter(ee.Filter.eq("ADM0_NAME", "Serbia"))
)

statistika = mean_annual_precip.reduceRegions(
    collection=opstine,
    reducer=ee.Reducer.mean(),
    scale=10000,
)

statistika.limit(5).getInfo()


{'type': 'FeatureCollection',
 'columns': {'ADM0_CODE': 'Integer',
  'ADM0_NAME': 'String',
  'ADM1_CODE': 'Integer',
  'ADM1_NAME': 'String',
  'ADM2_CODE': 'Integer',
  'ADM2_NAME': 'String',
  'DISP_AREA': 'String',
  'EXP2_YEAR': 'Integer',
  'STATUS': 'String',
  'STR2_YEAR': 'Integer',
  'Shape_Area': 'Float',
  'Shape_Leng': 'Float',
  'mean': 'Float',
  'system:index': 'String'},
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Polygon',
    'coordinates': [[[21.398298427884725, 42.75053444632896],
      [21.398441164027684, 42.74943748408664],
      [21.399471247863588, 42.74782324408276],
      [21.400055383639273, 42.74664606348634],
      [21.40086240266918, 42.74569180722377],
      [21.402111003555934, 42.74488918393282],
      [21.403943650010028, 42.74386362438134],
      [21.40607068518048, 42.743127872078034],
      [21.408121849870113, 42.742320752070654],
      [21.41003036987509, 42.741437859747656],
      [21.410690287294454, 42.7411479860547],
      [21